# ⚽Projet IA Football -  Fontanié Mathis & Hamoudi Malak- Modèle de Prédiction
Prédiction du résultat d’un match de football (victoire domicile, match nul, victoire extérieur) à l’aide de modèles supervisés : Random Forest, GridSearchCV et Arbre de Décision.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Affichage clair des tableaux 
pd.set_option('display.max_columns', None)


: 

### Chargement et fusion des données d'entraînement




In [ ]:
# Chargement des fichiers CSV
home_df = pd.read_csv("data/X_Train_Data/train_home_team_statistics_df.csv")
away_df = pd.read_csv("data/X_Train_Data/train_away_team_statistics_df.csv")
y_df = pd.read_csv("data/Y_train_1rknArQ.csv")

# Renommer les colonnes pour différencier home et away
home_df = home_df.add_suffix('_home')
away_df = away_df.add_suffix('_away')

# Remettre la colonne 'ID' pour la fusion
home_df.rename(columns={'ID_home': 'ID'}, inplace=True)
away_df.rename(columns={'ID_away': 'ID'}, inplace=True)

# Fusion des fichiers home et away sur ID
merged_df = pd.merge(home_df, away_df, on='ID')

# Fusion avec le fichier Y (resultat du match)
full_df = pd.merge(merged_df, y_df, on='ID')

#  dataframe final
print("Shape du dataframe final :", full_df.shape)
full_df.head()


## Nettoyage et préparation des données


In [ ]:
# Verifier le nombre de valeurs manquantes
nb_nan_par_colonne = full_df.isna().sum()
colonnes_nan = nb_nan_par_colonne[nb_nan_par_colonne > 0]

print("Nombre de colonnes avec NaN :", len(colonnes_nan))
print(colonnes_nan.sort_values(ascending=False))


In [ ]:
# Supprimer les colonnes non numériques 
colonnes_a_supprimer = [col for col in full_df.columns if 'LEAGUE' in col or 'TEAM_NAME' in col] + ['ID']
df_clean = full_df.drop(columns=colonnes_a_supprimer)

# Créer X les features
X = df_clean.drop(columns=['HOME_WINS', 'DRAW', 'AWAY_WINS'])
# Remplacement des NaN par la moyenne de chaque colonne
X = X.fillna(X.mean())

# Verifier qu'il ne reste plus de NaN
print("Y a-t-il encore des NaN ?", X.isna().sum().sum() > 0)


# Creer y : la variable cible codée 0 = HOME, 1 = DRAW, 2 = AWAY
y = df_clean[['HOME_WINS', 'DRAW', 'AWAY_WINS']].idxmax(axis=1).map({
    'HOME_WINS': 0,
    'DRAW': 1,
    'AWAY_WINS': 2
})

# Afficher la taille et la répartition des classes
print("Dimensions de X :", X.shape)
print("Répartition de y :\n", y.value_counts())


## Entraînement et comparaison de plusieurs modèles 

Dans cette cellule, on a :
- Séparerr les données en train/test (80/20)
- Entraîner un modèle Random Forest simple 
- Optimiser un Random Forest avec GridSearchCV
- Comparer les performances avec un Arbre de Décision

Les performances sont évaluées via l'accuracy, le rapport de classification et la matrice de confusion.


In [ ]:

#Split train/test (80% entraînement, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Création et entraînement du modèle Arbre de Décision
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

#  Prédiction sur le même X_test que RandomForest
y_pred_tree = tree_model.predict(X_test)

# Évaluation du modèle Arbre de Décision
print("\n ÉVALUATION DU MODELE ARBRE DE DECISION :")
print("Accuracy :", accuracy_score(y_test, y_pred_tree))
print("\nClassification report :\n", classification_report(y_test, y_pred_tree))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred_tree))

# Modèle Random Forest de base (non optimisé)
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
# Évaluation
print("ÉVALUATION DU MODELE RANDOM FOREST DE BASE")
print("Accuracy :", accuracy_score(y_test, y_pred_baseline))
print("\nClassification report :\n", classification_report(y_test, y_pred_baseline))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred_baseline))

# Dictionnaire des hyperparamètres à tester
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Création du modèle RandomForest
# Création du modèle de base
rf = RandomForestClassifier(random_state=42)
# GridSearch avec validation croisée 3-fold
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
# Entraînement sur X_train / y_train
grid_search.fit(X_train, y_train)

# Meilleur modèle trouvé
best_model = grid_search.best_estimator_

# Prédiction
y_pred = best_model.predict(X_test)

# Évaluation
print(" MEILLEUR MODELE TROUVÉ PAR GridSearchCV")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("\nClassification report :\n", classification_report(y_test, y_pred))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))



Nous avons testé trois modèles :

Random Forest de base (accuracy : 0.474)

Arbre de Décision (accuracy : 0.394)

Random Forest optimisé (accuracy : 0.483)



## Affichage clair des performances du modèle de base

In [ ]:
# Matrice de confusion du modèle Random Forest de base
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
labels = ['Home Win (0)', 'Draw (1)', 'Away Win (2)']


plt.figure(figsize=(6, 5))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Oranges', xticklabels=labels, yticklabels=labels)
plt.xlabel("Prédiction")
plt.ylabel("Vérité terrain")
plt.title("Matrice de confusion - Random Forest de base")
plt.tight_layout()
plt.show()

 ## Affichage clair des performances du modèle Arbre de Décision

In [ ]:
# Matrice de confusion du modèle Arbre de Décision
cm_tree = confusion_matrix(y_test, y_pred_tree)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
plt.xlabel("Prédiction")
plt.ylabel("Vérité terrain")
plt.title("Matrice de confusion - Arbre de Décision")
plt.tight_layout()
plt.show()

## Affichage visuel de la matrice de confusion du meilleur modèle (GridSearchCV)


In [ ]:

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel("Prédiction")
plt.ylabel("Vérité terrain")
plt.title("Matrice de confusion")
plt.tight_layout()
plt.show()


### Préparation du fichier X_test_final pour la prédiction



In [ ]:
# Charger les fichiers test
home_test_df = pd.read_csv("data/X_Test_Data/test_home_team_statistics_df.csv")
away_test_df = pd.read_csv("data/X_Test_Data/test_away_team_statistics_df.csv")

# Renommer les colonnes pour différencier home et away
home_test_df = home_test_df.rename(columns=lambda col: col + "_home" if col != "ID" else col)
away_test_df = away_test_df.rename(columns=lambda col: col + "_away" if col != "ID" else col)

# Fusionner sur ID
X_test_final = pd.merge(home_test_df, away_test_df, on="ID")

# Supprimer les colonnes inutiles
X_test_final = X_test_final.drop(columns=["LEAGUE_home", "TEAM_NAME_home", "LEAGUE_away", "TEAM_NAME_away"], errors="ignore")

# Remplacer les NaN par la moyenne comme dans X_train
X_test_final = X_test_final.fillna(X.mean())

# Vérifier la correspondance des colonnes
X_test_final = X_test_final[X.columns]  # mettre dans le même ordre que X


### Prédiction finale sur les données de test et génération du fichier `Y_pred_final.csv`



Nous avons choisi d'utiliser Random Forest optimisé comme modèle final, car il a obtenu la meilleure accuracy (~48.3%) sur notre jeu de test.

Il a donc été utilisé pour générer les prédictions finales sur les données X_test_final, selon le format attendu (ID, HOME_WINS, DRAW, AWAY_WINS)

In [ ]:
# Prédiction sur le vrai X_test (sans la colonne ID)
y_pred = best_model.predict(X_test_final)

# 0/1 au lieu de True/False
y_pred_bin = pd.get_dummies(y_pred)

# S'assurer que toutes les colonnes [0, 1, 2] existent
for col in [0, 1, 2]:
    if col not in y_pred_bin.columns:
        y_pred_bin[col] = 0

# Réordonner les colonnes et les renommer
y_pred_bin = y_pred_bin[[0, 1, 2]]
y_pred_bin.columns = ["HOME_WINS", "DRAW", "AWAY_WINS"]

# Ajouter la colonne ID correctement
ids = home_test_df["ID"].values
y_pred_bin.insert(0, "ID", ids)

# Convertir tous les bool en int 
y_pred_bin = y_pred_bin.astype(int)

# Sauvegarde dans le bon format
y_pred_bin.to_csv("Y_pred_final.csv", index=False)
